<a href="https://colab.research.google.com/github/Lourdes-Churacutipa/Repositorio-de-lourdes/blob/main/Proyecto_MLP_LOURDES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto Integrador — Ciencia de Datos Ambientales (UTEC)
## Modelo 2: Red Neuronal Perceptrón Multicapa (MLP) — Lourdes Churacutipa

**Instrucciones de uso:**
* Si estás trabajando sobre el notebook compartido donde ya corrieron las secciones 1–6 (y donde están los bloques de Julia/Rocío en `Proyecto__SVR_ROCIO.ipynb`), puedes pegar **directamente las celdas de las Secciones 2, 3 y 4** de este archivo al final de tu notebook en Colab.
* Si deseas correr este notebook de forma independiente (`Proyecto_MLP_Lourdes.ipynb`), ejecuta primero las celdas de las **Secciones 1 a 6** de `CDA_Piura_Proyecto_Integrador_final_CORREGIDO.ipynb` para tener en memoria `base_mensual` y `DRIVE`, y luego ejecuta de corrido todo este flujo.

## 0. Importación de Librerías y Conexión a Google Drive (`DATOS CDA`)

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.inspection import permutation_importance
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings('ignore', category=ConvergenceWarning)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/DATOS CDA'
except ImportError:
    DRIVE = './DATOS CDA'

if os.path.exists(DRIVE):
    print('Archivos en la carpeta DATOS CDA:', len(os.listdir(DRIVE)))
else:
    print('AVISO: Verifica que el acceso directo "DATOS CDA" esté en la raíz de Mi unidad.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archivos en la carpeta DATOS CDA: 1


## 1. Verificación de `base_mensual` (Secciones 1 a 6 del Notebook Base)
> **Nota:** Este bloque verifica que `base_mensual` (generada en las secciones 1 a 6 de `CDA_Piura_Proyecto_Integrador_final_CORREGIDO.ipynb` o `Proyecto__SVR_ROCIO.ipynb`) exista en memoria antes de iniciar el Paso 1.

In [ ]:
if 'base_mensual' not in globals():
    ruta_csv = os.path.join(DRIVE, 'base_mensual.csv')
    if os.path.exists(ruta_csv):
        base_mensual = pd.read_csv(ruta_csv, parse_dates=['fecha'])
        print('base_mensual cargada desde CSV:', base_mensual.shape)
    else:
        raise RuntimeError(
            'Ejecuta primero las Secciones 1 a 6 del notebook base para crear `base_mensual` '
            'con las columnas ["fecha", "mes", "pp_mm", "E", "ICEN", "ONI", "C"].'
        )
else:
    print('base_mensual verificada en memoria:', base_mensual.shape)

RuntimeError: Ejecuta primero las Secciones 1 a 6 del notebook base para crear `base_mensual` con las columnas ["fecha", "mes", "pp_mm", "E", "ICEN", "ONI", "C"].

---
## 2. Pasos 1 y 2: Tabla Común y Evaluación Común (Protocolo Fijo)
* Si ya ejecutaste las celdas de **Paso 1** y **Paso 2** más arriba en el notebook (por ejemplo, debajo de los bloques de Julia o Rocío), estas celdas garantizan que `tabla`, `X`, `y`, `evaluar_pliegues`, `res_base` y `pred_base` tengan exactamente los valores oficiales:
  * **Filas:** `231` (`1981-12` a `2020-02`)
  * **Conteo `mes_temp`:** `{1: 39, 2: 39, 3: 39, 4: 38, 5: 38, 6: 38}`
  * **RMSE medio de la línea base (Climatología):** `65.6 mm`

In [ ]:
# ==============================================================================
# PASO 1: LA TABLA COMÚN (NO MODIFICAR)
# ==============================================================================
base_mensual['fecha'] = pd.to_datetime(base_mensual['fecha'])

temp = {12: 1, 1: 2, 2: 3, 3: 4, 4: 5, 5: 6}  # mes calendario -> mes de temporada
tabla = base_mensual[['fecha', 'mes', 'pp_mm', 'E', 'ICEN', 'ONI', 'C']].copy()
tabla = tabla[tabla['mes'].isin(temp)]
tabla = tabla[(tabla['fecha'] >= '1981-12-01') & (tabla['fecha'] <= '2020-02-01')]
tabla['mes_temp'] = tabla['mes'].map(temp)

print('Nulos antes de quitarlos:', int(tabla.isna().sum().sum()))
tabla = tabla.dropna().sort_values('fecha').reset_index(drop=True)  # el orden por fecha es obligatorio

VARIABLES = ['E', 'ICEN', 'ONI', 'C', 'mes_temp']
X = tabla[VARIABLES]
y = tabla['pp_mm']

print('Filas:', len(tabla), '| periodo:', f"{tabla['fecha'].min():%Y-%m} a {tabla['fecha'].max():%Y-%m}")
print(tabla['mes_temp'].value_counts().sort_index().to_dict())

NameError: name 'base_mensual' is not defined

In [ ]:
# ==============================================================================
# PASO 2: LA EVALUACIÓN COMÚN (NO MODIFICAR)
# ==============================================================================
def metricas(y_real, y_pred):
    return {
        'RMSE': mean_squared_error(y_real, y_pred) ** 0.5,
        'MAE': mean_absolute_error(y_real, y_pred),
        'R2': r2_score(y_real, y_pred)
    }

def evaluar_pliegues(ajustar_predecir, nombre):
    """ajustar_predecir(X_tr, y_tr, X_te) debe devolver las predicciones para X_te."""
    filas, preds = [], []
    for k, (i_tr, i_te) in enumerate(TimeSeriesSplit(n_splits=5).split(X), 1):
        X_tr, y_tr, X_te, y_te = X.iloc[i_tr], y.iloc[i_tr], X.iloc[i_te], y.iloc[i_te]
        y_pred = ajustar_predecir(X_tr, y_tr, X_te)
        fila = {
            'modelo': nombre,
            'pliegue': k,
            'n_train': len(i_tr),
            'n_test': len(i_te),
            'periodo_test': f"{tabla.loc[i_te[0], 'fecha']:%Y-%m} a {tabla.loc[i_te[-1], 'fecha']:%Y-%m}"
        }
        fila.update(metricas(y_te, y_pred))
        filas.append(fila)
        preds.append(pd.DataFrame({
            'modelo': nombre,
            'pliegue': k,
            'fecha': tabla.loc[i_te, 'fecha'].values,
            'pp_obs': y_te.values,
            'pp_pred': y_pred
        }))
    return pd.DataFrame(filas), pd.concat(preds, ignore_index=True)

def climatologia(X_tr, y_tr, X_te):
    normal = y_tr.groupby(X_tr['mes_temp']).mean()  # lluvia media de cada mes, SOLO con el entrenamiento
    return X_te['mes_temp'].map(normal).values

res_base, pred_base = evaluar_pliegues(climatologia, 'Climatología')
display(res_base.round(2))
print('RMSE medio de la línea base:', round(res_base['RMSE'].mean(), 1), 'mm')

---
## 3. Paso 3: Modelo 2 — Red Neuronal Perceptrón Multicapa (`MLP`)
### Reglas metodológicas aplicadas:
1. **Escalado dentro de `Pipeline`:** `StandardScaler()` se ajusta únicamente con `X_tr` en cada corte de validación cruzada, evitando cualquier fuga de información del futuro.
2. **Búsqueda de hiperparámetros anidada:** `GridSearchCV` usa `cv=TimeSeriesSplit(n_splits=3)` y `scoring='neg_root_mean_squared_error'`.
3. **Control de sobreajuste en muestra pequeña ($N=231$):** Se evalúan arquitecturas compactas de 1 y 2 capas ocultas (`(16,)`, `(32,)`, `(16, 8)`, `(32, 16)`), funciones de activación no lineales (`relu`, `tanh`) y regularización L2 (`alpha`), fijando `random_state=42` y sin `early_stopping=True`.
4. **Restricción física:** Las predicciones negativas se recortan a `0 mm` con `np.maximum(pred, 0)`.

In [ ]:
# ==============================================================================
# PASO 3: MODELO 2 — RED NEURONAL MLP (LOURDES)
# ==============================================================================
REJILLA_MLP = {
    'mlpregressor__hidden_layer_sizes': [(16,), (32,), (16, 8), (32, 16)],
    'mlpregressor__activation': ['relu', 'tanh'],
    'mlpregressor__alpha': [0.01, 0.1, 1.0, 5.0]
}

mejores_mlp = []
modelos_entrenados_mlp = []

def mlp_ajustado(X_tr, y_tr, X_te):
    modelo = make_pipeline(
        StandardScaler(),
        MLPRegressor(max_iter=5000, random_state=42)
    )
    busqueda = GridSearchCV(
        estimator=modelo,
        param_grid=REJILLA_MLP,
        cv=TimeSeriesSplit(n_splits=3),
        scoring='neg_root_mean_squared_error',
        n_jobs=-1
    )
    busqueda.fit(X_tr, y_tr)
    mejores_mlp.append(busqueda.best_params_)
    modelos_entrenados_mlp.append(busqueda.best_estimator_)
    return np.maximum(busqueda.predict(X_te), 0)

res_mlp, pred_mlp = evaluar_pliegues(mlp_ajustado, 'MLP')
res_mlp['hiperparametros'] = [str(p) for p in mejores_mlp]

display(res_mlp.round(2))
print(f"RMSE medio MLP: {res_mlp['RMSE'].mean():.2f} ± {res_mlp['RMSE'].std():.2f} mm")
print(f"MAE medio MLP:  {res_mlp['MAE'].mean():.2f} ± {res_mlp['MAE'].std():.2f} mm")
print(f"R2 medio MLP:   {res_mlp['R2'].mean():.3f} ± {res_mlp['R2'].std():.3f}")

mejora_pct = (1 - res_mlp['RMSE'].mean() / res_base['RMSE'].mean()) * 100
pliegues_ganados = int((res_mlp['RMSE'].values < res_base['RMSE'].values).sum())
print(f"Mejora frente a climatología: {mejora_pct:.1f}% | Supera en {pliegues_ganados}/5 pliegues")

---
## 4. Importancia de Variables (Permutación) y Gráficos para Presentación
Calcula la importancia de cada predictor mediante permutación (`permutation_importance`) en los 5 pliegues de prueba y genera las figuras diagnósticas requeridas para la exposición (Sección 3.5).

In [ ]:
# Importancia de variables por permutación (incremento en RMSE al permutar cada variable)
importancias_pliegues = []
for k, (i_tr, i_te) in enumerate(TimeSeriesSplit(n_splits=5).split(X)):
    X_te, y_te = X.iloc[i_te], y.iloc[i_te]
    modelo_k = modelos_entrenados_mlp[k]
    perm = permutation_importance(
        modelo_k, X_te, y_te,
        n_repeats=30,
        random_state=42,
        scoring='neg_root_mean_squared_error'
    )
    importancias_pliegues.append(perm.importances_mean)

df_imp = pd.DataFrame(importancias_pliegues, columns=VARIABLES)
resumen_imp = pd.DataFrame({
    'Variable': VARIABLES,
    'Aumento_RMSE_medio_mm': df_imp.mean().values,
    'Desv_Std_mm': df_imp.std().values
}).sort_values('Aumento_RMSE_medio_mm', ascending=True)

display(resumen_imp.sort_values('Aumento_RMSE_medio_mm', ascending=False).round(2))

# Gráficos diagnósticos: Importancia de variables + Observado vs Predicho
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].barh(
    resumen_imp['Variable'],
    resumen_imp['Aumento_RMSE_medio_mm'],
    xerr=resumen_imp['Desv_Std_mm'],
    color='#2b5c8f', capsize=4
)
axes[0].axvline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('Importancia de Variables (Permutación — MLP)')
axes[0].set_xlabel('Incremento medio de RMSE (mm)')
axes[0].grid(axis='x', alpha=0.3)

axes[1].scatter(pred_mlp['pp_obs'], pred_mlp['pp_pred'], alpha=0.75, color='#d95f02', edgecolors='k', label='Meses de prueba (MLP)')
max_val = max(pred_mlp['pp_obs'].max(), pred_mlp['pp_pred'].max()) * 1.05
axes[1].plot([0, max_val], [0, max_val], 'k--', label='Predicción perfecta (1:1)')
axes[1].set_title(f"MLP: Observado vs. Predicho (RMSE: {res_mlp['RMSE'].mean():.1f} mm)")
axes[1].set_xlabel('Lluvia Observada PISCO (mm)')
axes[1].set_ylabel('Lluvia Predicha MLP (mm)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('MLP_diagnostico_presentacion.png', dpi=300)
plt.show()

---
## 5. Paso 4: Exportar los 3 Archivos `.csv` para Nicole (Formato Idéntico a XGBoost)
Genera exactamente los tres archivos requeridos en `DATOS CDA/RESULTADOS_MODELO_LOURDES`:
1. `MLP_metricas_pliegues.csv` (mismas columnas que `XGBoost_metricas_pliegues.csv`: `modelo, pliegue, n_train, n_test, periodo_test, RMSE, MAE, R2, hiperparametros`)
2. `MLP_resumen.csv` (media y desviación de RMSE, MAE, R² y `pliegues_supera_base`)
3. `MLP_predicciones.csv` (predicción mes a mes en los 5 periodos de prueba para climatología y MLP)

In [ ]:
# ==============================================================================
# PASO 4: EXPORTAR PARA NICOLE (EXACTO SEGÚN GUÍA COMÚN)
# ==============================================================================
PREFIJO = 'MLP'
res_mod, pred_mod = res_mlp, pred_mlp
CARPETA_MODELOS = os.path.join(DRIVE, 'RESULTADOS_MODELO_LOURDES')
os.makedirs(CARPETA_MODELOS, exist_ok=True)

# 1) Métricas por pliegue (climatología + modelo)
metricas_mod = pd.concat([res_base, res_mod], ignore_index=True)
metricas_mod.to_csv(os.path.join(CARPETA_MODELOS, f'{PREFIJO}_metricas_pliegues.csv'), index=False)

# 2) Resumen (media y desviación)
resumen = metricas_mod.groupby('modelo').agg(
    RMSE_medio=('RMSE', 'mean'),
    RMSE_desv=('RMSE', 'std'),
    MAE_medio=('MAE', 'mean'),
    R2_medio=('R2', 'mean')
).round(2)
resumen['pliegues_supera_base'] = np.nan
resumen.loc[PREFIJO, 'pliegues_supera_base'] = int((res_mod['RMSE'].values < res_base['RMSE'].values).sum())
resumen.to_csv(os.path.join(CARPETA_MODELOS, f'{PREFIJO}_resumen.csv'))
display(resumen)

# 3) Predicciones mes a mes
pd.concat([pred_base, pred_mod], ignore_index=True).to_csv(
    os.path.join(CARPETA_MODELOS, f'{PREFIJO}_predicciones.csv'),
    index=False
)

print('Guardado en', CARPETA_MODELOS, ':', sorted(os.listdir(CARPETA_MODELOS)))

# Verificación rápida de columnas idénticas a XGBoost_metricas_pliegues.csv
print('\nVista previa de MLP_metricas_pliegues.csv:')
display(metricas_mod)

---
## 6. (Opcional) Tabla Comparativa Directa en Notebook (XGBoost vs. SVR vs. MLP)
Si los resultados de Julia (`RESULTADOS_MODELO_JULIA`) y Rocío (`RESULTADOS_MODELO_ROCIO`) ya están guardados en `DATOS CDA`, esta celda los lee y muestra la comparación inmediata junto a tu modelo `MLP`.

In [ ]:
rutas_resumen = [
    os.path.join(DRIVE, 'RESULTADOS_MODELO_JULIA', 'XGBoost_resumen.csv'),
    os.path.join(DRIVE, 'RESULTADOS_MODELO_ROCIO', 'SVR_resumen.csv'),
    os.path.join(DRIVE, 'RESULTADOS_MODELO_LOURDES', 'MLP_resumen.csv')
]

tablas_resumen = []
for r in rutas_resumen:
    if os.path.exists(r):
        tablas_resumen.append(pd.read_csv(r))

if tablas_resumen:
    comparativa = pd.concat(tablas_resumen, ignore_index=True).drop_duplicates(subset=['modelo'])
    display(comparativa.sort_values('RMSE_medio'))
else:
    print('Ejecuta primero el Paso 4 para generar MLP_resumen.csv.')